# 03 — Silero VAD

Run Silero VAD on the 16 kHz clip and see which parts it marks as speech. Then change
`threshold` and re-run to see how it affects what counts as speech vs silence.
First run needs internet access once, to download the model via `torch.hub`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# ---- constants ----
FILE_16K = "../outputs/sample_16k.wav"  # Silero VAD expects 16 kHz (or 8 kHz) input
THRESHOLDS_TO_TRY = [0.2, 0.5, 0.8]  # lower = more sensitive (catches quieter/faster speech, more false positives)

# ---- load Silero VAD ----
model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad", model="silero_vad", force_reload=False, trust_repo=True
)
(get_speech_timestamps, _, read_audio, *_rest) = utils

wav = read_audio(FILE_16K, sampling_rate=16000)
duration_s = len(wav) / 16000
print(f"Loaded {FILE_16K}: {duration_s:.2f}s at 16000 Hz")

# ---- run VAD at a few thresholds and compare ----
fig, ax = plt.subplots(figsize=(12, 2.5 * len(THRESHOLDS_TO_TRY)))
results_by_threshold = {}

for i, threshold in enumerate(THRESHOLDS_TO_TRY):
    speech_segments = get_speech_timestamps(
        wav, model, sampling_rate=16000, threshold=threshold
    )
    results_by_threshold[threshold] = speech_segments

    print(f"\nthreshold={threshold}: {len(speech_segments)} speech segment(s)")
    for seg in speech_segments:
        start_s = seg["start"] / 16000
        end_s = seg["end"] / 16000
        print(f"  speech: {start_s:.2f}s -> {end_s:.2f}s")

# ---- plot the waveform with speech regions shaded, one row per threshold ----
time_axis = np.linspace(0, duration_s, len(wav))
fig, axes = plt.subplots(len(THRESHOLDS_TO_TRY), 1, figsize=(12, 3 * len(THRESHOLDS_TO_TRY)), sharex=True)
if len(THRESHOLDS_TO_TRY) == 1:
    axes = [axes]

for ax, threshold in zip(axes, THRESHOLDS_TO_TRY):
    ax.plot(time_axis, wav.numpy(), linewidth=0.5, color="gray")
    for seg in results_by_threshold[threshold]:
        ax.axvspan(seg["start"] / 16000, seg["end"] / 16000, color="orange", alpha=0.4)
    ax.set_title(f"threshold={threshold} (orange = detected speech)")
    ax.set_ylabel("amplitude")

axes[-1].set_xlabel("time (s)")
fig.tight_layout()
fig.savefig("../outputs/vad_thresholds.png", dpi=120)
print("\nSaved ../outputs/vad_thresholds.png")
plt.show()
